# IPL Intelligence — 03. Bowling, Matchups & Dismissals

Analyze bowling economy, wicket-taking, dot-ball pressure, dismissal modes, batter–bowler head-to-head records and sample sizes. This notebook distinguishes wickets recorded from wickets credited to the bowler.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "deliveries.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
df = pd.read_csv(DATA / "deliveries.csv", low_memory=False)
matches = pd.read_csv(DATA / "matches.csv", low_memory=False)
for c in ["start_date"]:
    if c in df: df[c] = pd.to_datetime(df[c], errors="coerce")
for c in ["runs_off_bat","extras","total_runs","legal_ball","is_wicket","is_bowler_credited_wicket","over_number"]:
    if c in df: df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
df["season"] = df["season"].astype(str)
print(f"Deliveries: {len(df):,} | Matches: {df.match_id.nunique():,} | Seasons: {df.season.nunique()}")

## 1. Bowling metrics

In [ ]:
d = df.copy()
for c in ["byes","legbyes","penalty","wides","noballs"]:
    if c not in d: d[c] = 0
    d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)
d["bowler_runs_conceded"] = d.total_runs - d.byes - d.legbyes - d.penalty
bow = d.groupby("bowler", as_index=False).agg(
    runs_conceded=("bowler_runs_conceded","sum"), legal_balls=("legal_ball","sum"),
    wickets=("is_bowler_credited_wicket","sum"), matches=("match_id","nunique"),
    dot_deliveries=("total_runs",lambda s:s.eq(0).sum())
).rename(columns={"bowler":"player"})
bow["overs"] = bow.legal_balls/6
bow["economy"] = np.where(bow.legal_balls>0,6*bow.runs_conceded/bow.legal_balls,np.nan)
bow["bowling_average"] = np.where(bow.wickets>0,bow.runs_conceded/bow.wickets,np.nan)
bow["bowling_strike_rate"] = np.where(bow.wickets>0,bow.legal_balls/bow.wickets,np.nan)
bow["dot_pct"] = np.where(bow.legal_balls>0,100*bow.dot_deliveries/bow.legal_balls,np.nan)
qualified_bowl = bow[bow.legal_balls >= 600]
display(qualified_bowl.sort_values("wickets",ascending=False).head(20).round(2))

## 2. Economy vs wicket-taking

In [ ]:
fig, ax = plt.subplots(figsize=(9,6)); ax.scatter(qualified_bowl.economy, qualified_bowl.wickets, s=np.maximum(qualified_bowl.legal_balls/100,12), alpha=.65); ax.set_xlabel("Economy rate"); ax.set_ylabel("Bowler-credited wickets"); ax.set_title("Economy vs wickets (bubble size ~ legal balls)"); fig.tight_layout()

## 3. Dismissal patterns

In [ ]:
dismissals = d[d.wicket_type.fillna("").astype(str).str.strip().ne("")].groupby("wicket_type",as_index=False).size().sort_values("size",ascending=False)
display(dismissals)
fig, ax = plt.subplots(figsize=(9,5)); ax.barh(dismissals.wicket_type[::-1],dismissals["size"][::-1]); ax.set_title("Recorded dismissal types"); ax.set_xlabel("Dismissals"); fig.tight_layout()

## 4. Batter vs bowler matchup explorer

In [ ]:
batter = "RG Sharma"
bowler = "JJ Bumrah"
matchup = d[d.striker.eq(batter) & d.bowler.eq(bowler)].copy()
print(f"{batter} vs {bowler}: {len(matchup)} recorded deliveries across {matchup.match_id.nunique()} matches")
print("Runs off bat:", int(matchup.runs_off_bat.sum()))
print("Legal balls:", int(matchup.legal_ball.sum()))
print("Dismissals of striker:", int(matchup.player_dismissed.eq(batter).sum()))
print("Recorded dismissal rows involving this batter:")
display(matchup.loc[matchup.player_dismissed.notna(), ["season","start_date","match_id","innings","ball","runs_off_bat","extras","wicket_type","player_dismissed"]])
if len(matchup):
    by_season=matchup.groupby("season",as_index=False).agg(deliveries=("match_id","size"),runs=("runs_off_bat","sum"),matches=("match_id","nunique"),dismissals=("player_dismissed",lambda s:s.eq(batter).sum()))
    display(by_season)

## 5. Matchup caution
Head-to-head summaries are descriptive, not a pure measure of skill: sample sizes, innings phase, venue, batting position, and player age/form are not controlled. A player dismissed once in a small sample should not be labelled as having a weakness.